In [0]:
# 1 Load Silver Feature Tables
user_behavior = spark.read.table(
"workspace.ecommerce_lakehouse.silver_user_behavior_features"
)

product_popularity = spark.read.table(
"workspace.ecommerce_lakehouse.silver_product_popularity_features"
)

rfm_features = spark.read.table(
"workspace.ecommerce_lakehouse.silver_user_rfm_features"
)

In [0]:
# Check Schemas:
user_behavior.printSchema()
product_popularity.printSchema()
rfm_features.printSchema()

In [0]:
# 2 Prepare Product Popularity Features .Because popularity table contains window timestamps, we simplify it.This creates a simple joinable feature.
from pyspark.sql.functions import col

product_popularity_features = product_popularity.select(
    col("product_id"),
    col("events_last_hour")
)

In [0]:
#3 Join Behavioral + RFM Features.. Join on user_id.
# Now you have: behavioral signals + customer value signals

user_features = user_behavior.join(
    rfm_features,
    on="user_id",
    how="left"
)

In [0]:
# 4 Create Training Labels ..We derive labels from behavior.
# Example rule: purchase > 0 → label = 1 ; otherwise → label = 0
from pyspark.sql.functions import when

training_df = user_features.withColumn(
    "label_purchase",
    when(col("purchases") > 0, 1).otherwise(0)
)

#For fraud detection:(This is synthetic but valid for demo.)
training_df = training_df.withColumn(
    "label_fraud",
    when(col("purchases") > 3, 1).otherwise(0)
)

In [0]:
#5 Select Final ML Columns. Select ML-ready features.Now this is a clean ML dataset.
gold_dataset = training_df.select(

"user_id",
"session_number",

"total_events",
"views",
"add_to_cart",
"purchases",

"avg_price",

"frequency",
"monetary_value",
"recency_days",

"label_purchase",
"label_fraud"

)

In [0]:
#6 Save Gold Table
gold_dataset.write.format("delta") \
.mode("overwrite") \
.saveAsTable(
"workspace.ecommerce_lakehouse.gold_ml_training_dataset"
)

# Verify Dataset, This is Ml ready data
display(
spark.read.table(
"workspace.ecommerce_lakehouse.gold_ml_training_dataset"
))